[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/02_bpe_desde_cero/02_proyecto_bpe_desde_cero.ipynb)

# Proyecto 02 · Tu BPE para Políglota
**Módulo 02 · BPE desde cero**

| | |
|---|---|
| **Nivel** | 🟡 Intermedio |
| **Duración estimada** | ≈ 2 h |
| **GPU recomendada** | Ninguna (CPU) |
| **Entregable** | tu `BPETokenizer` verificado contra `tiktoken` y HF `tokenizers`, y `poliglota_bpe.json` (merges) |

## Contexto de negocio
Políglota Labs va a indexar documentación en español, inglés y código. El CTO quiere **un tokenizer propio** (para controlar costes y para poder inspeccionarlo) y exige **evidencia de que está bien hecho**: que reproduce exactamente a `tiktoken` y a Hugging Face cuando se cargan sus merges, que comprime razonablemente cada idioma y que se sabe cuánto cuesta entrenar con un idioma y usarlo en otro.

**Conexión.** *Anterior (módulo 01):* tu texto llega normalizado a NFC (usa `unicodedata.normalize('NFC', t)`, o tu `normalizar` del módulo 01). *Siguiente (módulo 03):* compararás este BPE con WordPiece y Unigram sobre el mismo corpus, y el módulo 04 lo evaluará en fertilidad y paridad.

## Dataset
Corpus Políglota (`carga_corpus`, módulo 00): Wikipedia ES/EN + código Python; cadena **oficial → espejo → sintético** (la celda de carga imprime la fuente). Partición por documento: 80 % entrenamiento, 20 % prueba.

## Entregables y criterios de evaluación (rúbrica)

| # | Criterio | Umbral |
|---|---|---|
| R1 | `contar_palabras` pasa sus tests (regex GPT-2, bytes, frecuencias) | tests ✅ |
| R2 | `entrena_ingenuo` coincide con el incremental dado (60 merges sobre el corpus) y con el caso de mano | tests ✅ |
| R3 | `_encode_trozo`/`decode_bytes`: ida y vuelta sin pérdida en **todos** los documentos de prueba; con `max_id=256` el texto queda en bytes | 100 % |
| R4 | `a_tiktoken`: `tiktoken` con **tus** merges produce los mismos ids que tu `encode` | 100 % de ≥ 40 textos |
| R5 | HF `tokenizers` con **tus** merges produce los mismos ids | 100 % de ≥ 40 textos |
| R6 | Compresión en **prueba** (tokenizer entrenado en la mezcla ES+EN+código, vocabulario 1 000): bytes por token | ES ≥ **3.0**, EN ≥ **3.0**, código ≥ **1.9** (corpus sintético) |
| R7 | **Impuesto de idioma**: bytes/token en prueba ES del tokenizer entrenado solo en ES ÷ el entrenado solo en EN | ≥ **1.5**× (corpus sintético) |
| R8 | Barrido de vocabulario: bytes/token **no decrece** al crecer el vocabulario (ES, EN, código) | monótono |
| R9 | `poliglota_bpe.json` guardado (merges, vocabulario, regex y métricas) | ✅ |

Umbrales calibrados con **lo que mide la solución de referencia** (corpus **sintético**, `FAST_DEV_RUN=True`, vocabulario 1 000, semilla 42): ES = 3.29, EN = 3.33, código = 2.19 bytes/token; prima ES = 2.10×. Los umbrales tienen un margen de ≈ 0.3 (0.6 en la prima). **No pudimos medir la solución con Wikipedia real** en el entorno de autoría (Hugging Face bloqueado): con datos reales R6 y R7 **no están calibrados**; las comprobaciones estructurales (R1–R5, R8, R9) valen igual y la solución imprime sus valores para que compares.

## 0. Preparación (celdas dadas)

In [ ]:
!pip install -q numpy pandas matplotlib regex tiktoken tokenizers

In [ ]:
import heapq, json, math, pathlib, random, re, sys, time, unicodedata, collections
from collections import Counter, defaultdict
import numpy as np, pandas as pd, regex
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
FAST_DEV_RUN = True
VOCAB = 1000 if FAST_DEV_RUN else 4000
CHECKS: dict = {}

def chequea(nombre, fn):
    """Ejecuta fn() y registra ✅/❌/⏳ sin romper «Ejecutar todo»."""
    try:
        fn(); CHECKS[nombre] = True; print("✅", nombre)
    except NotImplementedError as e:
        CHECKS[nombre] = None; print("⏳ pendiente:", nombre, "→", e)
    except AssertionError as e:
        CHECKS[nombre] = False; print("❌", nombre, "→", e)
    except Exception as e:  # noqa: BLE001
        CHECKS[nombre] = False; print("💥", nombre, "→", type(e).__name__, e)

In [ ]:
# ── carga_corpus (1/3) ──
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── carga_corpus (2/3) ──
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus (3/3) ──
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
import regex

# Patrones de pre-tokenización copiados de tiktoken 0.14.0 (tiktoken_ext/openai_public.py; verificados el 25-oct-2026)
PAT_GPT2 = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}++| ?\p{N}++| ?[^\s\p{L}\p{N}]++|\s++$|\s+(?!\S)|\s"""
PAT_CL100K = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}++|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s++$|\s*[\r\n]|\s+(?!\S)|\s"""
PAT_O200K = "|".join([
    r"""[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]*[\p{Ll}\p{Lm}\p{Lo}\p{M}]+(?i:'s|'t|'re|'ve|'m|'ll|'d)?""",
    r"""[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]+[\p{Ll}\p{Lm}\p{Lo}\p{M}]*(?i:'s|'t|'re|'ve|'m|'ll|'d)?""",
    r"""\p{N}{1,3}""", r""" ?[^\s\p{L}\p{N}]+[\r\n/]*""", r"""\s*[\r\n]+""", r"""\s+(?!\S)""", r"""\s+"""])
PAT_ESPACIOS = r"""\s?\S+|\s+"""      # línea base SIN categorías: corta solo por espacios (puntuación pegada a la palabra)
PATRONES = {"gpt2": PAT_GPT2, "cl100k": PAT_CL100K, "o200k": PAT_O200K, "solo espacios": PAT_ESPACIOS}

In [ ]:
# ── Dado: fusión de un par en una palabra y entrenador INCREMENTAL de la lección ──
def fusiona_palabra(w, par, nuevo):
    a, b = par; out, i = [], 0
    while i < len(w):
        if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
            out.append(nuevo); i += 2
        else:
            out.append(w[i]); i += 1
    return out


def entrena_incremental(wc, n_merges):
    """Conteo INCREMENTAL: contadores de pares + índice invertido par -> palabras + heap con invalidación perezosa.
    Cada merge solo toca las palabras que contienen el par y ajusta los contadores de sus vecinos."""
    palabras, frec = [list(w) for w in wc], list(wc.values())
    stats, donde = Counter(), defaultdict(set)
    for i, (w, f) in enumerate(zip(palabras, frec)):
        for p in zip(w, w[1:]):
            stats[p] += f; donde[p].add(i)
    heap = [(-c, p) for p, c in stats.items()]; heapq.heapify(heap); merges = []
    while len(merges) < n_merges and heap:
        negc, par = heapq.heappop(heap)
        if stats.get(par, 0) != -negc:                       # entrada obsoleta: el contador cambió desde que se metió
            continue
        nuevo, (a, b), cambiados = 256 + len(merges), par, set(); merges.append(par)
        for i in list(donde[par]):
            w, f, out, j = palabras[i], frec[i], [], 0
            while j < len(w):
                if j < len(w) - 1 and w[j] == a and w[j + 1] == b:
                    if out:                                  # vecino izquierdo
                        stats[(out[-1], a)] -= f; stats[(out[-1], nuevo)] += f; donde[(out[-1], nuevo)].add(i)
                        cambiados |= {(out[-1], a), (out[-1], nuevo)}
                    if j + 2 < len(w):                       # vecino derecho
                        stats[(b, w[j + 2])] -= f; stats[(nuevo, w[j + 2])] += f; donde[(nuevo, w[j + 2])].add(i)
                        cambiados |= {(b, w[j + 2]), (nuevo, w[j + 2])}
                    stats[par] -= f; out.append(nuevo); j += 2
                else:
                    out.append(w[j]); j += 1
            palabras[i] = out
        del stats[par]; donde.pop(par, None)
        for p in cambiados - {par}:
            c = stats.get(p, 0)
            if c > 0:
                heapq.heappush(heap, (-c, p))
            else:
                stats.pop(p, None)
    return merges

In [ ]:
corpus = carga_corpus(fast_dev=FAST_DEV_RUN)
docs, source = corpus["docs"], corpus["source"]
nf = lambda t: unicodedata.normalize("NFC", t)
train = {l: [nf(d["text"]) for i, d in enumerate([d for d in docs if d["lang"] == l]) if i % 5 != 0] for l in ("es", "en", "code")}
test = {l: [nf(d["text"]) for i, d in enumerate([d for d in docs if d["lang"] == l]) if i % 5 == 0] for l in ("es", "en", "code")}
print(f"source = {source!r} | train: {sum(map(len, train.values()))} docs | test: {sum(map(len, test.values()))} docs")

## TODO 1 · Contar palabras
Implementa `contar_palabras(textos, pat_str=PAT_GPT2)`: parte cada texto con el regex, convierte cada trozo a **tupla de bytes UTF-8** y devuelve un `Counter` `{tupla: frecuencia}`. (Entrenar sobre palabras únicas ponderadas es lo que hace Sennrich y lo que hace tractable el entrenamiento.)

<details>
<summary>🪜 Pista 1</summary>

`regex.compile(pat_str).findall(texto)` devuelve los trozos; cada trozo `w` se convierte con `tuple(w.encode('utf-8'))`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`Counter(tuple(w.encode('utf-8')) for t in textos for w in pat.findall(t))`.

</details>

In [ ]:
def contar_palabras(textos, pat_str=PAT_GPT2):
    # TODO
    raise NotImplementedError("TODO 1: implementa contar_palabras")

def test_contar():
    wc = contar_palabras(["a b a", "ab"])
    assert wc == Counter({(97,): 1, (32, 98): 1, (32, 97): 1, (97, 98): 1}), f"obtuve {wc}"
    assert contar_palabras(["¿qué?"])[tuple("é".encode())] == 0 and tuple("qué".encode()) in contar_palabras(["¿qué?"])
chequea("R1 contar_palabras", test_contar)

## TODO 2 · Entrenador ingenuo
Implementa `entrena_ingenuo(wc, n_merges)`: en cada iteración **recuenta todos los pares** (ponderados por la frecuencia de la palabra), elige el de **mayor conteo y, ante empate, el menor** `(a, b)`, lo añade a `merges` y lo fusiona en todas las palabras con id nuevo `256 + k` (usa `fusiona_palabra`). Devuelve la lista de pares. Debe coincidir **exactamente** con `entrena_incremental`.

<details>
<summary>🪜 Pista 1</summary>

Mantén `palabras = [list(w) for w in wc]` y `frec = list(wc.values())`. Cada iteración: un `Counter` de pares con `zip(w, w[1:])`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Elección: `min(stats, key=lambda p: (-stats[p], p))`. Después `palabras = [fusiona_palabra(w, par, 256 + k) for w in palabras]`. Detente si `stats` está vacío.

</details>

In [ ]:
def entrena_ingenuo(wc, n_merges):
    # TODO
    raise NotImplementedError("TODO 2: implementa entrena_ingenuo")

def test_ingenuo():
    assert entrena_ingenuo(Counter({(97, 98): 3, (97, 97, 98): 1}), 1) == [(97, 98)], "(a,b) aparece 4 veces; (a,a), 1"
    wc = contar_palabras(train["es"][:40] + train["code"][:20])
    assert entrena_ingenuo(wc, 60) == entrena_incremental(wc, 60), "el ingenuo y el incremental deben dar los mismos merges"
chequea("R2 entrena_ingenuo", test_ingenuo)

## TODO 3 · Codificar y decodificar
Completa dos métodos de `BPETokenizer` (el resto está dado): `_encode_trozo(trozo, max_id=None)` y `decode_bytes(ids)`.
* `_encode_trozo`: convierte el trozo a ids de byte (`self.byte2id`), y **mientras haya algún par en `self.merges`**, fusiona el de **menor id resultante** (`merges[par]`) en todas sus apariciones (con `fusiona_palabra`). Si `max_id` no es `None`, no apliques merges con id `>= max_id`. Guarda el resultado en `self._cache`.
* `decode_bytes`: concatena `self.vocab[i]` (o los bytes del token especial) → `bytes`.

<details>
<summary>🪜 Pista 1</summary>

Bucle `while len(ids) > 1:`; candidatos `[(self.merges.get(p, 1 << 60), p) for p in zip(ids, ids[1:])]`; `rango, par = min(candidatos)`; si `rango == 1 << 60` (o `>= max_id`) sal del bucle.

</details>

<details>
<summary>🪜 Pista 2</summary>

`ids = fusiona_palabra(ids, par, rango)`. Para decodificar: `b''.join(self.vocab[i] if i in self.vocab else inv_especiales[i] for i in ids)` con `inv_especiales = {v: k.encode() for k, v in self.special.items()}`.

</details>

In [ ]:
class BPETokenizer:
    def __init__(self, pat_str=PAT_GPT2, vocab=None, merges=None, special_tokens=None):
        self.pat_str, self.pat = pat_str, regex.compile(pat_str)
        self.vocab = vocab or {i: bytes([i]) for i in range(256)}
        self.merges = merges or {}                    # (id_a, id_b) -> id nuevo (el id nuevo ES la prioridad)
        self.special = special_tokens or {}
        self.byte2id = {b: i for i, b in self.vocab.items() if len(b) == 1}
        self._cache = {}

    @classmethod
    def train(cls, textos, vocab_size, pat_str=PAT_GPT2):
        lista = entrena_incremental(contar_palabras(textos, pat_str), vocab_size - 256)
        vocab, merges = {i: bytes([i]) for i in range(256)}, {}
        for k, (a, b) in enumerate(lista):
            vocab[256 + k] = vocab[a] + vocab[b]; merges[(a, b)] = 256 + k
        return cls(pat_str, vocab, merges)

    def _encode_trozo(self, trozo, max_id=None):
        # TODO: aplica los merges por rango (menor id primero)
        raise NotImplementedError("TODO 3a: implementa _encode_trozo")

    def encode(self, texto, allowed_special=(), max_id=None):
        partes = regex.split("(" + "|".join(map(regex.escape, allowed_special)) + ")", texto) if allowed_special else [texto]
        ids = []
        for p in partes:
            if p in self.special and p in allowed_special: ids.append(self.special[p])
            else: ids += [i for t in self.pat.findall(p) for i in self._encode_trozo(t, max_id)]
        return ids

    def decode_bytes(self, ids):
        # TODO
        raise NotImplementedError("TODO 3b: implementa decode_bytes")

    def decode(self, ids): return self.decode_bytes(ids).decode("utf-8", errors="replace")

tok = None
try:
    tok = BPETokenizer.train(train["es"] + train["en"] + train["code"], VOCAB)
    tok.special = {"<|endoftext|>": len(tok.vocab)}
    print(f"vocabulario: {len(tok.vocab)} tokens | merges: {len(tok.merges)} | primeros tokens aprendidos:", [tok.vocab[i] for i in range(256, 264)])
except NotImplementedError as e:
    print("⏳ pendiente:", e)

def test_roundtrip():
    if tok is None: raise NotImplementedError('completa antes los TODO 1-3')
    todos = [t for l in test for t in test[l]]
    assert all(tok.decode(tok.encode(t)) == t for t in todos), "ida y vuelta fallida"
    assert all(len(tok.encode(t, max_id=256)) == len(t.encode()) for t in todos[:30]), "con max_id=256 debe haber 1 token por byte"
    assert tok.decode(tok.encode("a<|endoftext|>b", allowed_special={"<|endoftext|>"})) == "a<|endoftext|>b"
chequea("R3 encode/decode", test_roundtrip)

## TODO 4 · Verifica contra `tiktoken`
Implementa `a_tiktoken(tok)`: un `tiktoken.Encoding` con `pat_str=tok.pat_str`, `mergeable_ranks = {bytes del token: id}` y `special_tokens=dict(tok.special)`. Luego compara los ids de `tok.encode(t)` y `enc.encode(t, allowed_special="all")` sobre ≥ 40 textos variados (los dados + los tuyos).

<details>
<summary>🪜 Pista 1</summary>

`tiktoken.Encoding(name, pat_str=..., mergeable_ranks=..., special_tokens=...)`. Las claves de `mergeable_ranks` son `bytes`; el valor es el rango (aquí, el id).

</details>

<details>
<summary>🪜 Pista 2</summary>

`ranks = {tok.vocab[i]: i for i in tok.vocab}`. Si `len(ranks) != len(tok.vocab)` hay dos tokens con los mismos bytes y tiktoken no lo admite.

</details>

In [ ]:
import tiktoken
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, Regex

def a_tiktoken(tok, nombre="mi_bpe"):
    # TODO
    raise NotImplementedError("TODO 4: implementa a_tiktoken")

PRUEBAS = [t[:300] for l in test for t in test[l][:12]] + ["Hola, ¿cómo estás? 😀 你好\n\n  x", "naïve café 123456 ", "   ", "a\tb\r\nc  ", "don't DON'T", "x=3.14159;y=-42", "", "Año 2026"]

def test_tiktoken():
    if tok is None: raise NotImplementedError('completa antes los TODO 1-3')
    assert len(PRUEBAS) >= 40
    enc = a_tiktoken(tok)
    malos = [t for t in PRUEBAS if tok.encode(t) != enc.encode(t)]
    assert not malos, f"{len(malos)}/{len(PRUEBAS)} textos difieren, p. ej. {malos[0][:50]!r}"
chequea("R4 == tiktoken", test_tiktoken)

## TODO 5 · Verifica contra Hugging Face `tokenizers` (dado) y analiza
La función `hf_desde_mio` está dada (usa `bytes_to_unicode` de GPT-2 y un pre-tokenizer `Split(regex) + ByteLevel`). Solo tienes que ejecutar la comparación y **explicar** (en la celda de texto) por qué el *entrenador* de HF, a diferencia de su *codificador*, no reproduce tus merges.

In [ ]:
def bytes_to_unicode():
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs, n = bs[:], 0
    for b in range(256):
        if b not in bs: bs.append(b); cs.append(256 + n); n += 1
    return dict(zip(bs, map(chr, cs)))
B2U = bytes_to_unicode(); a_str = lambda b: "".join(B2U[x] for x in b)
GPT2_ORIG = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

def hf_desde_mio(tk, pat=GPT2_ORIG):
    vocab = {a_str(tk.vocab[i]): i for i in sorted(tk.vocab)}
    merges = [(a_str(tk.vocab[a]), a_str(tk.vocab[b])) for (a, b), _ in sorted(tk.merges.items(), key=lambda kv: kv[1])]
    hf = Tokenizer(models.BPE(vocab=vocab, merges=merges))
    hf.pre_tokenizer = pre_tokenizers.Sequence([pre_tokenizers.Split(Regex(pat), behavior="isolated"), pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False)])
    hf.decoder = decoders.ByteLevel(); return hf

def test_hf():
    if tok is None: raise NotImplementedError('completa antes los TODO 1-3')
    hf = hf_desde_mio(tok)
    malos = [t for t in PRUEBAS if tok.encode(t) != hf.encode(t).ids]
    assert not malos, f"{len(malos)}/{len(PRUEBAS)} textos difieren, p. ej. {malos[0][:50]!r}"
chequea("R5 == HF tokenizers", test_hf)

**✍️ Explica (2–4 líneas):** ¿por qué HF `BpeTrainer` y tu `entrena_incremental` pueden producir listas de merges distintas aunque el *codificador* coincida? ¿Qué parte del contrato del tokenizer decide? *(Pista: desempates.)*

> …

## TODO 6 · Evalúa: compresión, prima de tokens por idioma y barrido de vocabulario
Implementa `bytes_por_token(tk, textos, max_id=None)` (bytes UTF-8 totales ÷ tokens totales sobre `textos`). Después rellena `RES` con:
* `comp`: bytes/token de `tok` en prueba por idioma (`es`, `en`, `code`);
* `tax`: bytes/token en prueba ES del tokenizer entrenado **solo en ES** ÷ el entrenado **solo en EN** (mismo `VOCAB`);
* `barrido`: para `Vs = [256, 400, 600, 800, VOCAB]` y cada idioma, `bytes_por_token(tok, test[l], max_id=V)`.

<details>
<summary>🪜 Pista 1</summary>

Para la compresión, suma `len(t.encode())` y `len(tk.encode(t, max_id=max_id))` sobre los textos. El barrido no reentrena: usa `max_id` (el vocabulario de tamaño V es el prefijo de los merges).

</details>

<details>
<summary>🪜 Pista 2</summary>

`tax = bpt(tk_es, test['es']) / bpt(tk_en, test['es'])`, con `tk_es = BPETokenizer.train(train['es'], VOCAB)` y análogo para EN.

</details>

In [ ]:
def bytes_por_token(tk, textos, max_id=None):
    # TODO
    raise NotImplementedError("TODO 6a: implementa bytes_por_token")

RES = {}
try:
    Vs = [256, 400, 600, 800, VOCAB]
    tk_es, tk_en = BPETokenizer.train(train["es"], VOCAB), BPETokenizer.train(train["en"], VOCAB)
    RES = {"comp": {l: bytes_por_token(tok, test[l]) for l in test},
           "tax": bytes_por_token(tk_es, test["es"]) / bytes_por_token(tk_en, test["es"]),
           "barrido": {l: [bytes_por_token(tok, test[l], max_id=v) for v in Vs] for l in test}, "Vs": Vs}
    print({k: (round(v, 2) if isinstance(v, float) else {a: (round(b, 2) if isinstance(b, float) else [round(x, 2) for x in b]) for a, b in v.items()}) for k, v in RES.items() if k != "Vs"})
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    for l, c in zip(test, ("#1F6FEB", "#F28E2B", "#2CA02C")): ax.plot(Vs, RES["barrido"][l], "o-", color=c, label=l)
    ax.set_xlabel("tamaño del vocabulario"); ax.set_ylabel("bytes por token (prueba)"); ax.set_title(f"Compresión vs vocabulario (corpus {source})"); ax.legend(); plt.tight_layout(); plt.show()
except NotImplementedError as e:
    print("⏳ pendiente:", e)

> 👀 **Qué debes observar.** En $V=256$ (solo bytes) las tres curvas valen exactamente 1 byte/token; después suben rápido y se aplanan (rendimientos decrecientes, como en la lección §9.1). El **código** queda por debajo de ES/EN (más símbolos y combinaciones raras). Y la prima `tax` > 1 significa que, para el mismo texto en español, el tokenizer entrenado solo en español comprime más que el entrenado en inglés: la referencia mide ≈ 2.1× con corpus sintético.

## TODO 7 · Guardar tu tokenizer
Implementa `guardar_bpe(tok, path='poliglota_bpe.json')`: guarda un JSON con `pat_str`, `merges` (lista de pares `[id_a, id_b]` **en orden**), `vocab_size`, `special_tokens`, `source` y `metricas` (`RES['comp']`, `RES['tax']`). Debe poder reconstruirse: añade `cargar_bpe(path)` que devuelva un `BPETokenizer` con el mismo `encode`.

<details>
<summary>🪜 Pista 1</summary>

Con la lista ordenada de merges puedes reconstruir `vocab` (`vocab[256+k] = vocab[a] + vocab[b]`) y `merges`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`sorted(tok.merges.items(), key=lambda kv: kv[1])` te da los pares en orden de creación.

</details>

In [ ]:
def guardar_bpe(tok, path="poliglota_bpe.json"):
    # TODO
    raise NotImplementedError("TODO 7a: implementa guardar_bpe")

def cargar_bpe(path="poliglota_bpe.json"):
    # TODO
    raise NotImplementedError("TODO 7b: implementa cargar_bpe")

def test_json():
    guardar_bpe(tok); tok2 = cargar_bpe()
    assert all(tok.encode(t) == tok2.encode(t) for t in PRUEBAS) and len(tok2.vocab) == len(tok.vocab)
    assert {"pat_str", "merges", "vocab_size", "special_tokens", "source", "metricas"} <= set(json.loads(pathlib.Path("poliglota_bpe.json").read_text()))
chequea("R9 poliglota_bpe.json", test_json)

## Rúbrica: comprobación automática

In [ ]:
UMBRALES = {'es': 3.0, 'en': 3.0, 'code': 1.9, 'tax': 1.5}   # medidos con la solución (ver la tabla; solo corpus SINTÉTICO)

def rubrica():
    for n, e in CHECKS.items(): print({True: "✅", False: "❌", None: "⏳"}[e], n)
    if RES:
        mono = all(all(b >= a - 1e-9 for a, b in zip(v, v[1:])) for v in RES["barrido"].values())
        print("✅" if mono else "❌", "R8 barrido monótono:", {l: [round(x, 2) for x in v] for l, v in RES["barrido"].items()})
        if source == "synthetic":
            u = UMBRALES
            for nombre, v, t in (("R6 ES", RES["comp"]["es"], u["es"]), ("R6 EN", RES["comp"]["en"], u["en"]), ("R6 código", RES["comp"]["code"], u["code"]), ("R7 prima de tokens", RES["tax"], u["tax"])):
                print("✅" if v >= t else "❌", f"{nombre} = {v:.2f} (≥ {t})")
        else:
            print("ℹ️ corpus REAL: R6 y R7 no están calibrados; reporta tus valores:", {k: round(v, 2) for k, v in RES["comp"].items()}, "prima ES", round(RES["tax"], 2))
rubrica()

## Retos extra (nivel experto)
1. **Mejora la codificación a $O(n\log n)$** con una lista doblemente enlazada y un heap de pares adyacentes; mide la aceleración con trozos largos (URLs, líneas de código sin espacios) frente a tu `_encode_trozo`.
2. **Carga `cl100k_base`/`o200k_base`** (si `tiktoken` puede descargarlos) con la receta de `desde_tiktoken` de la lección y verifica tu `encode` con ellos sobre Políglota; compara tokens/carácter por idioma.
3. **Regex propio para Políglota**: parte de `PAT_O200K`, añade una regla que no parta `snake_case` en el código y mide bytes/token en código antes y después.
4. **BPE-dropout** (Provilkov et al., 2019): en `_encode_trozo`, ignora cada merge con probabilidad $p$ y observa cuánto varía la segmentación de una misma palabra entre llamadas.
5. **Tokens sub-entrenados:** cuenta cuántos tokens de tu vocabulario no aparecen nunca en el conjunto de prueba y cuántos aparecen < 3 veces en entrenamiento; relaciónalo con el módulo 05.

## Reflexión
* Tu tokenizer, entrenado en la mezcla, rinde peor en cada idioma que el especialista. ¿Qué proporción de datos por idioma elegirías para Políglota y cómo lo decidirías (coste en tokens vs calidad del retrieval)?
* ¿Qué ocurre si en producción el indexador usa tu tokenizer con NFC y el cliente envía texto NFD? ¿Dónde lo arreglas?
* El módulo 03 compara BPE con Unigram y WordPiece: ¿qué métricas, además de bytes/token, usarías para decidir?

---
# ⛔ SPOILER — intenta resolverlo primero
Debajo está la solución de referencia, completa y ejecutable. Si «Ejecutar todo» llega hasta aquí, estas celdas **sobrescriben** tus funciones con las de referencia.

In [ ]:
# ── SOLUCIÓN · TODO 1 y 2 ──
import heapq
from collections import Counter, defaultdict


def contar_palabras(textos, pat_str=PAT_GPT2):
    """Corpus -> {palabra (tupla de bytes): frecuencia}. Entrenar sobre palabras únicas ponderadas es lo que hace Sennrich."""
    pat = regex.compile(pat_str)
    return Counter(tuple(w.encode("utf-8")) for t in textos for w in pat.findall(t))


def fusiona_palabra(w, par, nuevo):
    a, b = par; out, i = [], 0
    while i < len(w):
        if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
            out.append(nuevo); i += 2
        else:
            out.append(w[i]); i += 1
    return out


def entrena_ingenuo(wc, n_merges, tie="lex"):
    """O(M · U): recuenta TODOS los pares en cada iteración. tie='lex' (menor par) o 'first' (primer visto, como tiktoken._educational)."""
    palabras, frec, merges = [list(w) for w in wc], list(wc.values()), []
    for k in range(n_merges):
        stats = Counter()
        for w, f in zip(palabras, frec):
            for p in zip(w, w[1:]):
                stats[p] += f
        if not stats:
            break
        par = min(stats, key=lambda p: (-stats[p], p)) if tie == "lex" else max(stats, key=stats.get)
        merges.append(par)
        palabras = [fusiona_palabra(w, par, 256 + k) for w in palabras]
    return merges

In [ ]:
# ── SOLUCIÓN · TODO 3 ──
class BPETokenizer:
    """Byte-level BPE. vocab: id -> bytes. merges: (id_a, id_b) -> id nuevo; el id nuevo ES la prioridad (menor = antes)."""

    def __init__(self, pat_str=PAT_GPT2, vocab=None, merges=None, special_tokens=None):
        self.pat_str, self.pat = pat_str, regex.compile(pat_str)
        self.vocab = vocab or {i: bytes([i]) for i in range(256)}
        self.merges = merges or {}
        self.special = special_tokens or {}
        self.byte2id = {b: i for i, b in self.vocab.items() if len(b) == 1}
        self._cache = {}

    @classmethod
    def train(cls, textos, vocab_size, pat_str=PAT_GPT2, tie="lex"):
        wc = contar_palabras(textos, pat_str)
        lista = entrena_incremental(wc, vocab_size - 256) if tie == "lex" else entrena_ingenuo(wc, vocab_size - 256, tie)
        vocab, merges = {i: bytes([i]) for i in range(256)}, {}
        for k, (a, b) in enumerate(lista):
            vocab[256 + k] = vocab[a] + vocab[b]; merges[(a, b)] = 256 + k
        return cls(pat_str, vocab, merges)

    def _encode_trozo(self, trozo: str, max_id=None):
        clave = (trozo, max_id)
        if clave not in self._cache:
            ids = [self.byte2id[bytes([b])] for b in trozo.encode("utf-8")]
            while len(ids) > 1:
                cand = [(self.merges.get(p, 1 << 60), p) for p in zip(ids, ids[1:])]
                rango, par = min(cand)                       # el par de MENOR id = el que se aprendió antes
                if rango == 1 << 60 or (max_id is not None and rango >= max_id):
                    break
                ids = fusiona_palabra(ids, par, rango)
            self._cache[clave] = ids
        return self._cache[clave]

    def encode(self, texto: str, allowed_special=(), max_id=None) -> list[int]:
        partes = [texto]
        if allowed_special:
            partes = regex.split("(" + "|".join(map(regex.escape, allowed_special)) + ")", texto)
        ids = []
        for p in partes:
            if p in self.special and p in allowed_special:
                ids.append(self.special[p])
            else:
                ids += [i for t in self.pat.findall(p) for i in self._encode_trozo(t, max_id)]
        return ids

    def decode_bytes(self, ids) -> bytes:
        inv = {v: k.encode() for k, v in self.special.items()}
        return b"".join(self.vocab[i] if i in self.vocab else inv[i] for i in ids)

    def decode(self, ids) -> str:
        return self.decode_bytes(ids).decode("utf-8", errors="replace")

tok = BPETokenizer.train(train["es"] + train["en"] + train["code"], VOCAB)
tok.special = {"<|endoftext|>": len(tok.vocab)}
print(f"vocabulario: {len(tok.vocab)} tokens | merges: {len(tok.merges)} | primeros tokens:", [tok.vocab[i] for i in range(256, 264)])

In [ ]:
# ── SOLUCIÓN · TODO 4 ──
def a_tiktoken(tok, nombre="mi_bpe"):
    ranks = {tok.vocab[i]: i for i in sorted(tok.vocab)}
    assert len(ranks) == len(tok.vocab), "hay dos tokens con los mismos bytes: tiktoken no lo admite"
    return tiktoken.Encoding(nombre, pat_str=tok.pat_str, mergeable_ranks=ranks, special_tokens=dict(tok.special))

In [ ]:
# ── SOLUCIÓN · TODO 6 y 7 ──
def bytes_por_token(tk, textos, max_id=None):
    nb_ = sum(len(t.encode("utf-8")) for t in textos); nt = sum(len(tk.encode(t, max_id=max_id)) for t in textos)
    return nb_ / nt

Vs = [256, 400, 600, 800, VOCAB]
tk_es, tk_en = BPETokenizer.train(train["es"], VOCAB), BPETokenizer.train(train["en"], VOCAB)
RES = {"comp": {l: bytes_por_token(tok, test[l]) for l in test}, "tax": bytes_por_token(tk_es, test["es"]) / bytes_por_token(tk_en, test["es"]),
       "barrido": {l: [bytes_por_token(tok, test[l], max_id=v) for v in Vs] for l in test}, "Vs": Vs}
print({l: round(v, 2) for l, v in RES["comp"].items()}, "| prima ES:", round(RES["tax"], 2))

def guardar_bpe(tok, path="poliglota_bpe.json"):
    merges = [list(p) for p, _ in sorted(tok.merges.items(), key=lambda kv: kv[1])]
    d = {"pat_str": tok.pat_str, "merges": merges, "vocab_size": len(tok.vocab), "special_tokens": tok.special, "source": source,
         "metricas": {"comp": RES["comp"], "tax": RES["tax"]}}
    pathlib.Path(path).write_text(json.dumps(d, ensure_ascii=False)); return d

def cargar_bpe(path="poliglota_bpe.json"):
    d = json.loads(pathlib.Path(path).read_text()); vocab, merges = {i: bytes([i]) for i in range(256)}, {}
    for k, (a, b) in enumerate(d["merges"]):
        vocab[256 + k] = vocab[a] + vocab[b]; merges[(a, b)] = 256 + k
    return BPETokenizer(d["pat_str"], vocab, merges, d["special_tokens"])

for nombre, fn in [("R1 contar_palabras", test_contar), ("R2 entrena_ingenuo", test_ingenuo), ("R3 encode/decode", test_roundtrip),
                   ("R4 == tiktoken", test_tiktoken), ("R5 == HF tokenizers", test_hf), ("R9 poliglota_bpe.json", test_json)]:
    chequea(nombre, fn)
rubrica()

> **📌 Valores de referencia de la solución.** Corpus **sintético**, `FAST_DEV_RUN=True`, vocabulario 1 000, semilla 42: bytes/token en prueba ES 3.29, EN 3.33, código 2.19; prima de tokens (ES-ES ÷ EN-ES) 2.10×; `tiktoken` y HF con tus merges: 100 % de coincidencia. Con datos **reales** tus números serán distintos (no pudimos medirlos en el entorno de autoría). Guarda `poliglota_bpe.json`: los módulos 03 y 04 lo usan como punto de comparación.